# F02: Warehouse Dimensional Load

Generate a retail star schema and load it into a Microsoft Fabric Warehouse using `FabricSqlDatabaseWriter`.

## What You'll Learn
- Generate retail data and inspect the star schema output
- Use `FabricSqlDatabaseWriter` with `create_insert` mode to load dimension and fact tables
- Generate SQL DDL with `create_ddl()` and bulk inserts with `to_sql()` / `to_sql_inserts()`
- Understand Fabric Warehouse limitations (no IDENTITY columns, no PK constraints enforced)

## Prerequisites
- Python 3.10+
- `sqllocks-spindle[fabric-sql]` installed (includes `pyodbc` and `azure-identity`)
- ODBC Driver 18 for SQL Server installed
- A Fabric Warehouse endpoint (or Azure SQL Database for local testing)
- `az login` completed (for Azure CLI authentication)

## Time Estimate
~15 minutes

In [ ]:
# %pip install sqllocks-spindle[fabric-sql]

## Step 1: Generate Retail Data

**What we're about to do:** Generate a retail dataset at small scale and inspect the tables that Spindle produces.

**Why this matters:** Before loading to a warehouse, you need to understand the schema -- column types, primary keys, and foreign key relationships. Spindle schemas carry this metadata, which drives DDL generation.

In [ ]:
from sqllocks_spindle import Spindle, RetailDomain

spindle = Spindle()
result = spindle.generate(domain=RetailDomain(), scale="small", seed=42)

print(result.summary())
print(f"\nTables: {result.table_names}")
print(f"FK integrity check: {len(result.verify_integrity())} errors")

## Step 2: Configure the FabricSqlDatabaseWriter

**What we're about to do:** Set up the `FabricSqlDatabaseWriter` with a connection string pointing to your Fabric Warehouse. Authentication uses Azure CLI by default -- just run `az login` before executing this notebook.

**Key concepts:**
- `auth_method="cli"` -- Uses your Azure CLI token (best for local dev)
- `auth_method="msi"` -- Uses Managed Identity (best for Fabric Notebooks)
- `auth_method="spn"` -- Uses Service Principal (best for CI/CD)
- `auth_method="sql"` -- Uses SQL auth with username/password (for on-prem SQL Server)

**Gotcha:** Fabric Warehouse does NOT support `IDENTITY` columns or enforced `PRIMARY KEY` constraints. Spindle's DDL generator handles this automatically -- it emits plain `INT NOT NULL` instead of `INT IDENTITY(1,1)`.

In [ ]:
from sqllocks_spindle.fabric.sql_database_writer import FabricSqlDatabaseWriter

# Replace with your Fabric Warehouse SQL endpoint
CONNECTION_STRING = (
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=YOUR_WAREHOUSE.datawarehouse.fabric.microsoft.com;"
    "Database=YOUR_WAREHOUSE;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
)

writer = FabricSqlDatabaseWriter(
    connection_string=CONNECTION_STRING,
    auth_method="cli",  # Change to "msi" in Fabric Notebooks
)

# Uncomment to test the connection:
# assert writer.test_connection(), "Connection failed -- check your endpoint and az login"
print("Writer configured (connection not tested yet -- uncomment test_connection above)")

## Step 3: Preview the Generated DDL

**What we're about to do:** Use `create_ddl()` to generate the T-SQL `CREATE TABLE` statements that Spindle will execute. This lets you inspect the DDL before running it.

**What to expect:** One `CREATE TABLE` per Spindle table, with column types inferred from the schema metadata (not just pandas dtypes). No `IDENTITY` or `PRIMARY KEY` constraints -- Fabric Warehouse doesn't enforce them.

In [ ]:
# Generate DDL without executing -- preview what will be created
ddl = writer.create_ddl(result, schema_name="dbo", dialect="tsql")
print(ddl[:3000])  # Show first 3000 chars
print(f"\n... ({len(ddl)} total characters of DDL)")

## Step 4: Generate SQL INSERT Statements

**What we're about to do:** Use `to_sql()` to write INSERT statements to `.sql` files. This is useful for offline review, version control, or loading via other tools.

**Tip:** The `to_sql_inserts()` method batches inserts and handles NULL values, datetime formatting, and string escaping automatically.

In [ ]:
from pathlib import Path

sql_dir = Path("warehouse_sql_output")
sql_files = result.to_sql(sql_dir)

for f in sql_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name} ({size_kb:.1f} KB)")

# Preview the first INSERT file
first_file = sql_files[0]
with open(first_file) as fh:
    content = fh.read()
print(f"\n--- Preview of {first_file.name} (first 1000 chars) ---")
print(content[:1000])

## Step 5: Write to Fabric Warehouse (create_insert mode)

**What we're about to do:** Execute `writer.write()` with `mode="create_insert"` to DROP existing tables, CREATE new ones, and INSERT all data. Tables are written in dependency order (parents before children).

**Gotcha:** This cell requires a live Fabric Warehouse connection. Update the `CONNECTION_STRING` above and uncomment the code below to run it.

**What to expect:** A `WriteResult` with per-table row counts, elapsed time, and any errors.

In [ ]:
# Uncomment the block below when you have a live Fabric Warehouse connection.
# The writer handles: DROP (reverse order) -> CREATE -> INSERT (dependency order).

# write_result = writer.write(
#     result,
#     schema_name="dbo",
#     mode="create_insert",   # DROP + CREATE + INSERT (full reset)
#     batch_size=1000,        # Rows per INSERT batch
# )
#
# print(write_result.summary())
# print(f"\nSuccess: {write_result.success}")

# For reference, write modes available:
print("Available write modes:")
print("  create_insert   - DROP + CREATE + INSERT (full reset)")
print("  insert_only     - INSERT into existing tables (no DDL)")
print("  truncate_insert - TRUNCATE + INSERT (keep schema, reset data)")
print("  append          - INSERT without truncating (for incremental loads)")

## What's Next?

You've seen how to generate retail data, preview DDL, export SQL INSERT files, and load everything to a Fabric Warehouse.

- **F03: SQL Database OLTP Simulation** -- Use `insert_only` and `append` modes to simulate daily incremental loads against a Fabric SQL Database
- **F06: Semantic Model Builder** -- Generate a `.bim` semantic model file on top of this warehouse data, with auto-wired relationships and DAX measures
- **F01: Medallion Architecture** -- Build a full Bronze/Silver/Gold pipeline with chaos injection and validation gates